In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS dev")

for schema_name in ["bronze", "silver", "gold", "landing"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS dev.{schema_name}")

spark.sql("CREATE VOLUME IF NOT EXISTS dev.landing.taxi_drop")

print("pantry ready")

In [0]:
spark.sql("SHOW CATALOGS").show()
spark.sql("SHOW SCHEMAS IN dev").show()

In [0]:
VOLUME_PATH = "/Volumes/dev/landing/taxi_drop"

trips = spark.table("samples.nyctaxi.trips")

days = ["2016-01-01", "2016-01-02", "2016-01-03"]

for day in days:
    one_day = trips.filter(f"date(tpep_pickup_datetime) = '{day}'")
    (one_day
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("header", "true")
        .csv(f"{VOLUME_PATH}/dt={day}"))
    print(f"wrote {day}: {one_day.count()} rows")

In [0]:
files = dbutils.fs.ls(VOLUME_PATH)
for f in files:
    print(f.name)